# Taller 1 — Procesamiento de datos olímpicos con Apache Spark

**Curso:** Gestión de datos y tecnologías Big Data
**Grupo:** 1
**Integrantes:** Yerko Gallardo, Sebastián Rojas, José Ignacio Rodríguez

**Objetivo:** procesar el registro histórico de deportistas y resultados olímpicos con PySpark, utilizando RDDs, DataFrames y Spark SQL.

**Fuente de datos:** archivos entregados en el curso, ubicados en la carpeta `data/` del proyecto (`deportista.csv`, `deportista2.csv`, `resultados.csv`, `evento.csv`, `equipo.csv`, `juegos.json`).

**Estructura:** las secciones siguen los puntos de la consigna.

| Punto de la consigna | Sección | Responsable |
|---|---|---|
| 1. Configuración inicial del entorno | 1 | Yerko Gallardo |
| 2. RDDs: creación y unión (incluye el DataFrame `deportista`) | 2 | Yerko Gallardo |
| 3. RDDs: transformaciones | 3 | Yerko Gallardo |
| 4. DataFrames: creación e integración | 4 | Sebastián Rojas |
| 5. Paralelismo | 5 | Sebastián Rojas |
| 6. Inspección del DataFrame | 6 | Sebastián Rojas |
| 7. Columnas calculadas | 7 | José Ignacio Rodríguez |
| 8. Agregaciones requeridas (Spark SQL) | 8 | José Ignacio Rodríguez |
| 9. Presentación mediante video | 9 (archivo aparte) | Todos |
| Conclusiones, fuentes y cierre | 10 | José Ignacio Rodríguez |

## 1. Configuración inicial del entorno

El proyecto se gestiona con **uv**: la versión de Python y los paquetes quedan fijados en `pyproject.toml`, `uv.lock` y `.python-version`, lo que permite reproducir el mismo entorno en cualquier equipo.
**Java** es una dependencia de ejecución de Spark (el motor corre en la JVM) y se instala fuera de uv.

| Componente | Versión |
|---|---|
| Python | 3.14 |
| PySpark (incluye Apache Spark) | 4.3.0 |
| Java (JDK) | 17 o superior (Eclipse Temurin 17 o 21) |

**Preparación del proyecto:**

```bash
git clone https://github.com/joserodriguezc/gestiondatosytecnologiasbigdata.git
cd gestiondatosytecnologiasbigdata
uv sync --locked
uv run jupyter lab    # o VS Code, seleccionando el kernel .venv (Python 3.14)
```

La celda 1.1 completa la instalación en el propio notebook y no requiere intervención manual. Si PySpark y Java ya están instalados, los reutiliza; si faltan, los instala: PySpark con pip, y Java con `apt-get` en Colab o Linux con permisos de root, o con `install-jdk` en la carpeta del usuario en Windows y macOS (no requiere administrador). Luego define las variables de entorno. Las versiones efectivas se imprimen en la celda 1.2.

La sesión se crea en modo clásico (no Spark Connect), ya que el taller requiere RDDs y `SparkContext`. En Windows nativo, ver la advertencia de `README.md`.

### 1.1 Instalación de Spark y Java y variables de entorno

- **Spark:** se instala como el paquete `pyspark`, que incluye Apache Spark. Si ya está en el entorno (uv), no se reinstala.
- **Java:** primero se reutiliza un JDK instalado, en este orden: `JAVA_HOME`, `java` del PATH y las rutas habituales de Windows, macOS y Linux. Se acepta el primero con versión 17 o superior, el mínimo de Spark 4. Solo si no hay ninguno se instala un JDK 17: OpenJDK con `apt-get` en Colab o Linux con root, o Temurin con `install-jdk` en la carpeta del usuario (`~/.jdk`) en los demás casos, sin permisos de administrador. Si la instalación falla, la celda se detiene con un mensaje.
- **Variables de entorno:** `JAVA_HOME` (y su `bin` en el PATH), `PYSPARK_PYTHON`, `PYSPARK_DRIVER_PYTHON` y `SPARK_LOCAL_IP`.

In [1]:
import os, sys, platform, shutil, subprocess, re, glob, importlib.util
from pathlib import Path

VERSION_PYSPARK = "4.3.0"
JAVA_MINIMA = 17
SISTEMA = platform.system()
EJECUTABLE_JAVA = "java.exe" if SISTEMA == "Windows" else "java"


def instalar_paquete(paquete):
    """Instala un paquete en el intérprete actual, con pip o, en entornos uv sin pip, con uv."""
    if importlib.util.find_spec("pip"):
        comando = [sys.executable, "-m", "pip", "install", "-q", paquete]
    elif shutil.which("uv"):
        comando = ["uv", "pip", "install", "-q", "--python", sys.executable, paquete]
    else:
        raise RuntimeError(f"No hay pip ni uv para instalar {paquete}")
    subprocess.run(comando, check=True)


# --- Spark ---------------------------------------------------------------
if importlib.util.find_spec("pyspark") is None:
    print(f"PySpark no está instalado: se instala pyspark=={VERSION_PYSPARK}")
    instalar_paquete(f"pyspark=={VERSION_PYSPARK}")
else:
    print("PySpark ya está instalado en el entorno; no se reinstala.")


# --- Java ----------------------------------------------------------------
def version_java(java_home):
    """Versión mayor del JDK en java_home, o None si no hay un java ejecutable."""
    java = Path(java_home) / "bin" / EJECUTABLE_JAVA
    if not java.is_file():
        return None
    try:
        salida = subprocess.run([str(java), "-version"], capture_output=True, text=True, timeout=60).stderr
    except (OSError, subprocess.SubprocessError):
        return None
    coincidencia = re.search(r'version "(\d+)(?:\.(\d+))?', salida)
    if not coincidencia:
        return None
    mayor = int(coincidencia.group(1))
    return int(coincidencia.group(2) or 0) if mayor == 1 else mayor   # "1.8" corresponde a Java 8


def home_de_jdk(ruta):
    """En macOS el JDK descomprimido guarda su home en Contents/Home."""
    ruta = Path(ruta)
    return ruta / "Contents" / "Home" if (ruta / "Contents" / "Home").is_dir() else ruta


def candidatos_java():
    if os.environ.get("JAVA_HOME"):
        yield "JAVA_HOME", os.environ["JAVA_HOME"]
    java_en_path = shutil.which("java")
    if java_en_path:
        yield "PATH", Path(os.path.realpath(java_en_path)).parent.parent
    if SISTEMA == "Darwin":
        consulta = subprocess.run(["/usr/libexec/java_home", "-v", f"{JAVA_MINIMA}+"], capture_output=True, text=True)
        if consulta.returncode == 0:
            yield "java_home de macOS", consulta.stdout.strip()
    patrones = {
        "Windows": [r"C:\Program Files\Eclipse Adoptium\*", r"C:\Program Files\Java\*", r"C:\Program Files\Microsoft\jdk-*"],
        "Darwin": ["/Library/Java/JavaVirtualMachines/*"],
        "Linux": ["/usr/lib/jvm/*"],
    }.get(SISTEMA, []) + [str(Path.home() / ".jdk" / "*")]   # instalaciones de install-jdk
    for patron in patrones:
        for ruta in sorted(glob.glob(patron), reverse=True):
            yield "ruta habitual", ruta


def seleccionar_java():
    revisados = set()
    for origen, ruta in candidatos_java():
        java_home = home_de_jdk(os.path.realpath(ruta))
        if java_home in revisados:
            continue
        revisados.add(java_home)
        version = version_java(java_home)
        if version is not None and version >= JAVA_MINIMA:
            return origen, str(java_home), version
        print(f"  Descartado ({origen}): {java_home} -> versión {version}")
    return None


def instalar_java():
    if SISTEMA == "Linux" and shutil.which("apt-get") and os.geteuid() == 0:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", f"openjdk-{JAVA_MINIMA}-jdk-headless"], check=True)
    else:
        if importlib.util.find_spec("jdk") is None:
            instalar_paquete("install-jdk")
        import jdk
        print("  Temurin instalado en:", jdk.install(str(JAVA_MINIMA)))


seleccion = seleccionar_java()
if seleccion is None:
    print(f"No se encontró Java {JAVA_MINIMA} o superior: se instala un JDK {JAVA_MINIMA}.")
    try:
        instalar_java()
    except Exception as error:
        raise RuntimeError(f"No fue posible instalar Java automáticamente ({error}). Instale Temurin "
                           f"{JAVA_MINIMA} desde https://adoptium.net y defina JAVA_HOME.") from error
    seleccion = seleccionar_java()
    if seleccion is None:
        raise RuntimeError("Java se instaló, pero no se encontró un JDK utilizable.")

origen_java, java_home, version_jdk = seleccion
print(f"Java {version_jdk} encontrado mediante {origen_java}.")

# --- Variables de entorno ------------------------------------------------
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = str(Path(java_home) / "bin") + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

for variable in ["JAVA_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON", "SPARK_LOCAL_IP"]:
    print(f"{variable:22s}= {os.environ[variable]}")

PySpark ya está instalado en el entorno; no se reinstala.
Java 17 encontrado mediante JAVA_HOME.
JAVA_HOME             = /opt/java/openjdk
PYSPARK_PYTHON        = /opt/venv/bin/python
PYSPARK_DRIVER_PYTHON = /opt/venv/bin/python
SPARK_LOCAL_IP        = 127.0.0.1


### 1.2 Verificación del intérprete, versiones y Java

In [2]:
import pyspark

print("Sistema    :", platform.system(), platform.machine())
print("Intérprete :", sys.executable)
print("Python     :", platform.python_version())
print("PySpark    :", pyspark.__version__)
print("JAVA_HOME  :", os.environ.get("JAVA_HOME", "NO DEFINIDO"))

java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java       :")
print(java.stderr.strip())   # java -version escribe en stderr

Sistema    : Linux x86_64
Intérprete : /opt/venv/bin/python
Python     : 3.14.7
PySpark    : 4.3.0
JAVA_HOME  : /opt/java/openjdk
Java       :
openjdk version "17.0.21" 2026-10-20
OpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)
OpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)


### 1.3 SparkSession y SparkContext

La sesión usa las variables de entorno definidas en la celda 1.1: `JAVA_HOME` indica el JDK, y `PYSPARK_PYTHON` y `PYSPARK_DRIVER_PYTHON` apuntan al intérprete del proyecto, para que los procesos Python de Spark usen el mismo entorno.

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Taller1_Spark")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

print("Tipo de sesión     :", type(spark).__module__)   # pyspark.sql.session = modo clásico
print("Versión Spark      :", spark.version)
print("Master             :", sc.master)
print("Nombre aplicación  :", sc.appName)
print("Paralelismo default:", sc.defaultParallelism)
print("Spark UI           :", sc.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/04 22:40:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Tipo de sesión     : pyspark.sql.session
Versión Spark      : 4.3.0
Master             : local[*]
Nombre aplicación  : Taller1_Spark
Paralelismo default: 8
Spark UI           : http://localhost:4040


### 1.4 Prueba mínima de RDD y DataFrame

In [4]:
rdd_prueba = sc.parallelize(range(1, 11), 2)
print("RDD -> suma:", rdd_prueba.sum(), "| particiones:", rdd_prueba.getNumPartitions())

df_prueba = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "letra"])
df_prueba.show()
df_prueba.printSchema()

RDD -> suma: 55 | particiones: 2


+---+-----+
| id|letra|
+---+-----+
|  1|    a|
|  2|    b|
+---+-----+

root
 |-- id: long (nullable = true)
 |-- letra: string (nullable = true)



## 2. RDDs: creación y unión

Primero se inspeccionan los archivos de deportistas (2.1 a 2.4); luego se crean los RDDs `deportista` y `deportista2`, se unen en `deportistaTotal` y este se convierte en el DataFrame `deportista` (2.5 a 2.8).

Los datos de deportistas vienen repartidos en dos archivos CSV con la misma estructura de siete columnas:

| Columna | Contenido | Tipo |
|---|---|---|
| `deportista_id` | identificador del deportista (clave usada en `resultados.csv`) | entero |
| `nombre` | nombre del deportista | texto |
| `genero` | 1 = hombre, 2 = mujer | entero |
| `edad` | edad en años | entero |
| `altura` | altura en centímetros | decimal |
| `peso` | peso en kilogramos | decimal |
| `equipo_id` | identificador del equipo (clave de `equipo.csv`) | entero |

Antes de crear los RDDs se revisa el contenido crudo de cada archivo: separador, presencia de encabezado, uso de comillas y valores ausentes.

### 2.1 Ubicación de los archivos

In [5]:
from pathlib import Path

DIR_DATOS = Path("data")
RUTA_DEPORTISTA = DIR_DATOS / "deportista.csv"
RUTA_DEPORTISTA2 = DIR_DATOS / "deportista2.csv"

for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    estado = f"{ruta.stat().st_size:,} bytes" if ruta.exists() else "NO ENCONTRADO"
    print(f"{str(ruta):22s} {estado}")

faltantes = [str(r) for r in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2) if not r.exists()]
if faltantes:
    raise FileNotFoundError(f"Faltan archivos de entrada en la carpeta data/: {faltantes}")

data/deportista.csv    2,832,265 bytes
data/deportista2.csv   2,874,875 bytes


### 2.2 Contenido crudo

Se muestran las primeras líneas tal como las lee Spark con `textFile`. Se usa `repr` para dejar visibles las comillas y los caracteres de fin de línea.

In [6]:
for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    lineas = sc.textFile(str(ruta))
    print(f"--- {ruta.name}: {lineas.count():,} líneas")
    for linea in lineas.take(5):
        print("   ", repr(linea))
    print()

--- deportista.csv: 67,786 líneas
    '1,A Dijiang,1,24,180,80,199'
    '2,A Lamusi,1,23,170,60,199'
    '3,Gunnar Nielsen Aaby,1,24,0,0,273'
    '4,Edgar Lindenau Aabye,1,34,0,0,278'
    '5,Christine Jacoba Aaftink,2,21,185,82,705'



--- deportista2.csv: 67,785 líneas
    '67787,Lee BongJu,1,27,167,56,970'
    '67788,Lee BuTi,1,23,164,54,203'
    '67789,Anthony N. Buddy Lee,1,34,172,62,1096'
    '67790,Alfred A. Butch Lee Porter,1,19,186,80,825'
    '67791,Lee ByeongGu,1,22,175,68,970'



### 2.3 Reglas de lectura

- **Separación de campos:** cada línea se divide con el lector CSV de Python, que respeta los nombres entre comillas aunque contengan comas.
- **Encabezado:** solo la primera línea de un archivo puede ser encabezado; se reconoce porque su primer campo no es numérico. Se retira y se informa.
- **Valores ausentes:** los campos vacíos o con `NA` se interpretan como desconocidos (`None`).
- **Campos vacíos sobrantes:** si una línea trae más de siete campos y los sobrantes están vacíos (coma final de más), se conservan los siete primeros. Estos casos se cuentan y se muestran.
- **Registros inválidos:** una línea con un número de campos distinto de siete o con valores numéricos imposibles de convertir no se descarta en silencio: se separa en un RDD de inválidos y se informa con su motivo.
- **Ceros:** se conservan tal como vienen en los RDDs para mantener la trazabilidad con la fuente; su tratamiento se define al construir el DataFrame.

In [7]:
import csv

COLUMNAS = ["deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id"]
VALORES_AUSENTES = {"", "NA", "NaN", "null", "None"}


def separar_campos(linea):
    """Divide una línea CSV respetando comillas (nombres que contienen comas)."""
    return next(csv.reader([linea.rstrip("\r")]))


def sobran_campos_vacios(campos):
    """True si la línea trae más de siete campos y los sobrantes están vacíos (coma final extra)."""
    return len(campos) > len(COLUMNAS) and all(c.strip() == "" for c in campos[len(COLUMNAS):])


def normalizar_campos(campos):
    """Elimina los campos vacíos sobrantes al final de la línea; el resto queda igual."""
    return campos[:len(COLUMNAS)] if sobran_campos_vacios(campos) else campos


def es_encabezado(campos):
    return len(campos) > 0 and not campos[0].strip().isdigit()


def a_entero(valor):
    valor = valor.strip()
    if valor in VALORES_AUSENTES:
        return None
    numero = float(valor)
    if not numero.is_integer():
        raise ValueError(f"'{valor}' no es entero")
    return int(numero)


def a_decimal(valor):
    valor = valor.strip()
    return None if valor in VALORES_AUSENTES else float(valor)


def convertir_registro(campos):
    """Devuelve (registro_tipado, None) si la línea es válida, o (campos, motivo) si no lo es."""
    if len(campos) != len(COLUMNAS):
        return campos, f"tiene {len(campos)} campos"
    try:
        d_id, nombre, genero, edad, altura, peso, equipo_id = campos
        registro = (a_entero(d_id), nombre.strip(), a_entero(genero), a_entero(edad),
                    a_decimal(altura), a_decimal(peso), a_entero(equipo_id))
    except ValueError as error:
        return campos, f"valor no numérico ({error})"
    if registro[0] is None:
        return campos, "deportista_id vacío"
    return registro, None


def leer_deportistas(lineas):
    """A partir de un RDD de líneas devuelve (validos, invalidos, encabezado)."""
    indexadas = lineas.zipWithIndex()
    encabezado = (indexadas
                  .filter(lambda par: par[1] == 0 and es_encabezado(separar_campos(par[0])))
                  .map(lambda par: par[0]))
    convertidos = (indexadas
                   .filter(lambda par: not (par[1] == 0 and es_encabezado(separar_campos(par[0]))))
                   .map(lambda par: convertir_registro(normalizar_campos(separar_campos(par[0])))))
    validos = convertidos.filter(lambda r: r[1] is None).map(lambda r: r[0])
    invalidos = convertidos.filter(lambda r: r[1] is not None)
    return validos, invalidos, encabezado

### 2.4 Diagnóstico de calidad de cada archivo

Para cada archivo se cuenta el encabezado, los registros válidos e inválidos, los valores ausentes y los ceros de las columnas numéricas, y la codificación de `genero`.

In [8]:
def diagnosticar(ruta):
    lineas = sc.textFile(str(ruta))
    validos, invalidos, encabezado = leer_deportistas(lineas)
    validos.cache()

    total_lineas = lineas.count()
    n_encabezado = encabezado.count()
    n_validos = validos.count()
    n_invalidos = invalidos.count()

    print(f"=== {ruta.name}")
    print(f"Líneas leídas       : {total_lineas:,}")
    print(f"Encabezado          : {encabezado.first() if n_encabezado else 'no tiene'}")
    reparadas = lineas.filter(lambda l: sobran_campos_vacios(separar_campos(l)))
    print(f"Registros válidos   : {n_validos:,}")
    print(f"  con campos vacíos sobrantes reparados: {reparadas.count():,}")
    for linea in reparadas.take(5):
        print("    ", repr(linea))
    print(f"Registros inválidos : {n_invalidos:,}")
    for campos, motivo in invalidos.take(5):
        print(f"    {motivo}: {campos}")
    print(f"Conciliación        : {n_encabezado} + {n_validos:,} + {n_invalidos:,} = "
          f"{n_encabezado + n_validos + n_invalidos:,} ({'OK' if n_encabezado + n_validos + n_invalidos == total_lineas else 'NO CUADRA'})")

    ids = validos.map(lambda r: r[0])
    print(f"Rango deportista_id : {ids.min():,} a {ids.max():,} | ids distintos: {ids.distinct().count():,}")

    print(f"{'columna':10s} {'ausentes':>9s} {'ceros':>9s} {'mínimo':>8s} {'máximo':>8s}")
    for i, columna in enumerate(COLUMNAS):
        if columna in ("deportista_id", "nombre"):
            continue
        valores = validos.map(lambda r, i=i: r[i])
        presentes = valores.filter(lambda v: v is not None)
        print(f"{columna:10s} {valores.filter(lambda v: v is None).count():9,} "
              f"{presentes.filter(lambda v: v == 0).count():9,} {presentes.min():8} {presentes.max():8}")

    print("Codificación genero :", dict(sorted(validos.map(lambda r: r[2]).countByValue().items(),
                                              key=lambda kv: (kv[0] is None, kv[0]))))
    con_comillas = lineas.filter(lambda l: '"' in l).count()
    print(f"Líneas con comillas : {con_comillas:,}")
    validos.unpersist()
    print()


for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    diagnosticar(ruta)

=== deportista.csv
Líneas leídas       : 67,786
Encabezado          : no tiene
Registros válidos   : 67,786
  con campos vacíos sobrantes reparados: 1


     '16587,Vladimr Bil,1,24,191,92,67,'
Registros inválidos : 0


Conciliación        : 0 + 67,786 + 0 = 67,786 (OK)


Rango deportista_id : 1 a 67,786 | ids distintos: 67,786
columna     ausentes     ceros   mínimo   máximo


genero             0         0        1        2
edad               0     3,235        0       96


altura             0    17,435      0.0    223.0
peso               0    17,949      0.0    214.0


... (4 outputs left out)

Rango deportista_id : 67,787 a 135,571 | ids distintos: 67,785
columna     ausentes     ceros   mínimo   máximo
genero             0         0        1        2


edad               0     3,133        0       97
altura             0    16,481      0.0    226.0


peso               0    16,936      0.0    198.0
equipo_id          0        79        0     1184
Codificación genero : {1: 50210, 2: 17575}


Líneas con comillas : 0



**Hallazgos de la inspección**

| | `deportista.csv` | `deportista2.csv` |
|---|---|---|
| Líneas | 67.786 | 67.785 |
| Encabezado | no tiene | no tiene |
| Fin de línea | CRLF (Windows) | LF |
| Rango de `deportista_id` | 1 a 67.786 | 67.787 a 135.571 |
| Registros inválidos | 0 | 0 |
| Campos vacíos sobrantes reparados | 1 | 0 |
| Edad = 0 | 3.235 | 3.133 |
| Altura = 0 | 17.435 | 16.481 |
| Peso = 0 | 17.949 | 16.936 |
| `equipo_id` = 0 | 64 | 79 |
| Género 1 / 2 | 51.380 / 16.406 | 50.210 / 17.575 |

- Ninguno de los dos archivos trae encabezado y ninguno usa comillas; aun así, el lector respeta comillas por si un nombre contuviera comas. El fin de línea CRLF de `deportista.csv` se elimina al leer.
- Los dos archivos son complementarios: los identificadores son continuos del 1 al 135.571, sin repetidos ni vacíos entre ellos.
- La línea del deportista **16587** (`16587,Vladimr Bil,1,24,191,92,67,`) termina con una coma de más, que genera un octavo campo vacío. El registro está completo, por lo que se conservan sus siete campos en lugar de descartarlo.
- `genero` usa exclusivamente los códigos 1 (hombre) y 2 (mujer).
- Hay valores **0** en `edad`, `altura` y `peso`. Una edad, altura o peso igual a cero no es posible en un deportista: se trata de datos no registrados en la fuente. En los RDDs se conservan sin cambios y en el DataFrame se convierten a nulo.
- 143 deportistas tienen `equipo_id = 0`, código que no existe en `equipo.csv`; se mantienen y se informan en la integración con las demás tablas.
- Algunos nombres muestran caracteres perdidos en la fuente (por ejemplo, `Vladimr Bil` o `Piotr ya`, donde faltan letras con diacríticos); se conservan tal como vienen.

### 2.5 Creación de los RDDs `deportista` y `deportista2`

Se crea el RDD `deportista` a partir de `deportista.csv` con **seis particiones** y el RDD `deportista2` a partir de `deportista2.csv`. Luego se unen en `deportistaTotal`.

`textFile` recibe un número **mínimo** de particiones; si Spark genera más de seis, se reducen con `coalesce(6)`, que no redistribuye los datos entre nodos.

In [9]:
lineas_deportista = sc.textFile(str(RUTA_DEPORTISTA), minPartitions=6)
deportista, invalidos_deportista, encabezado_deportista = leer_deportistas(lineas_deportista)
if deportista.getNumPartitions() > 6:
    deportista = deportista.coalesce(6)
deportista = deportista.cache()

print("RDD deportista")
print("  Particiones          :", deportista.getNumPartitions())
print("  Registros por partición:", deportista.glom().map(len).collect())
print(f"  Registros válidos    : {deportista.count():,}")
print(f"  Registros inválidos  : {invalidos_deportista.count():,}")
print("  Muestra:")
for registro in deportista.take(5):
    print("   ", registro)

RDD deportista
  Particiones          : 6
  Registros por partición: [11396, 11236, 11095, 11243, 11424, 11392]


  Registros válidos    : 67,786
  Registros inválidos  : 0
  Muestra:
    (1, 'A Dijiang', 1, 24, 180.0, 80.0, 199)
    (2, 'A Lamusi', 1, 23, 170.0, 60.0, 199)
    (3, 'Gunnar Nielsen Aaby', 1, 24, 0.0, 0.0, 273)
    (4, 'Edgar Lindenau Aabye', 1, 34, 0.0, 0.0, 278)
    (5, 'Christine Jacoba Aaftink', 2, 21, 185.0, 82.0, 705)


In [10]:
lineas_deportista2 = sc.textFile(str(RUTA_DEPORTISTA2))
deportista2, invalidos_deportista2, encabezado_deportista2 = leer_deportistas(lineas_deportista2)
deportista2 = deportista2.cache()

print("RDD deportista2")
print("  Particiones          :", deportista2.getNumPartitions())
print(f"  Registros válidos    : {deportista2.count():,}")
print(f"  Registros inválidos  : {invalidos_deportista2.count():,}")
print("  Muestra:")
for registro in deportista2.take(5):
    print("   ", registro)

RDD deportista2
  Particiones          : 2


  Registros válidos    : 67,785
  Registros inválidos  : 0
  Muestra:
    (67787, 'Lee BongJu', 1, 27, 167.0, 56.0, 970)
    (67788, 'Lee BuTi', 1, 23, 164.0, 54.0, 203)
    (67789, 'Anthony N. Buddy Lee', 1, 34, 172.0, 62.0, 1096)
    (67790, 'Alfred A. Butch Lee Porter', 1, 19, 186.0, 80.0, 825)
    (67791, 'Lee ByeongGu', 1, 22, 175.0, 68.0, 970)


### 2.6 Unión en `deportistaTotal` y cantidad de registros

`union` concatena ambos RDDs sin eliminar duplicados, por lo que el número de particiones del resultado es la suma de las particiones de ambos. Se verifica que el total cuadre con la suma de los registros válidos de cada archivo y que ningún `deportista_id` se repita entre los dos archivos.

In [11]:
deportistaTotal = deportista.union(deportista2).cache()

n_deportista = deportista.count()
n_deportista2 = deportista2.count()
n_total = deportistaTotal.count()
n_ids_distintos = deportistaTotal.map(lambda r: r[0]).distinct().count()

print("RDD deportistaTotal")
print("  Particiones              :", deportistaTotal.getNumPartitions(),
      f"({deportista.getNumPartitions()} + {deportista2.getNumPartitions()})")
print(f"  deportista + deportista2 : {n_deportista:,} + {n_deportista2:,} = {n_deportista + n_deportista2:,}")
print(f"  deportistaTotal          : {n_total:,}")
print(f"  deportista_id distintos  : {n_ids_distintos:,}")

assert n_total == n_deportista + n_deportista2, "La unión no conserva todos los registros"
print("  Conciliación             : OK" if n_ids_distintos == n_total
      else f"  Atención: {n_total - n_ids_distintos:,} identificadores repetidos entre archivos")

RDD deportistaTotal
  Particiones              : 8 (6 + 2)
  deportista + deportista2 : 67,786 + 67,785 = 135,571
  deportistaTotal          : 135,571
  deportista_id distintos  : 135,571
  Conciliación             : OK


### 2.7 Conversión a DataFrame `deportista`

`deportistaTotal` se convierte en el DataFrame `deportista` con las siete columnas y un esquema explícito, de modo que cada columna tenga un tipo numérico o de texto utilizable en las etapas siguientes. La conversión se hace en este punto, como indica la consigna; por eso los nombres del DataFrame conservan su escritura original, y la conversión a mayúsculas del punto 3 se aplica al RDD `deportistaTotal`.

- `edad`, `altura` y `peso` iguales a 0 se convierten a **nulo**, porque representan datos no registrados. Así no distorsionan promedios, mínimos ni el cálculo de IMC.
- `genero` se mantiene con su codificación original (1 = hombre, 2 = mujer).
- `equipo_id` se mantiene sin cambios, incluido el valor 0, para revisarlo en la integración con `equipo.csv`.

El nombre `deportista` pasa a designar el DataFrame; el RDD original se conserva en `rdd_deportista`.

In [12]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

rdd_deportista = deportista   # referencia al RDD original antes de reutilizar el nombre

esquema_deportista = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("genero", IntegerType(), True),
    StructField("edad", IntegerType(), True),
    StructField("altura", DoubleType(), True),
    StructField("peso", DoubleType(), True),
    StructField("equipo_id", IntegerType(), True),
])


def cero_a_nulo(valor):
    return None if valor == 0 else valor


filas = deportistaTotal.map(
    lambda r: (r[0], r[1], r[2], cero_a_nulo(r[3]), cero_a_nulo(r[4]), cero_a_nulo(r[5]), r[6])
)
deportista = spark.createDataFrame(filas, schema=esquema_deportista)

print(f"Filas del DataFrame deportista: {deportista.count():,} (deportistaTotal: {deportistaTotal.count():,})")
print("Columnas:", deportista.columns)
deportista.printSchema()
deportista.show(10, truncate=False)

Filas del DataFrame deportista: 135,571 (deportistaTotal: 135,571)
Columnas: ['deportista_id', 'nombre', 'genero', 'edad', 'altura', 'peso', 'equipo_id']
root
 |-- deportista_id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)

+-------------+-----------------------------+------+----+------+----+---------+
|deportista_id|nombre                       |genero|edad|altura|peso|equipo_id|
+-------------+-----------------------------+------+----+------+----+---------+
|1            |A Dijiang                    |1     |24  |180.0 |80.0|199      |
|2            |A Lamusi                     |1     |23  |170.0 |60.0|199      |
|3            |Gunnar Nielsen Aaby          |1     |24  |NULL  |NULL|273      |
|4            |Edgar Lindenau Aabye         |1     |34  |NULL  |NULL|278      |
|5 

Traceback (most recent call last):
  File "/opt/venv/lib/python3.14/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 264, in manager
    code = worker(sock, authenticated)
  File "/opt/venv/lib/python3.14/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


### 2.8 Valores nulos por columna

In [13]:
deportista.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in deportista.columns]
).show()

deportista.describe("edad", "altura", "peso").show()

+-------------+------+------+----+------+-----+---------+
|deportista_id|nombre|genero|edad|altura| peso|equipo_id|
+-------------+------+------+----+------+-----+---------+
|            0|     0|     0|6368| 33916|34885|        0|
+-------------+------+------+----+------+-----+---------+



+-------+------------------+------------------+-----------------+
|summary|              edad|            altura|             peso|
+-------+------------------+------------------+-----------------+
|  count|            129203|            101655|           100686|
|   mean|24.447977214151376|176.31540996507795|71.96177224241701|
| stddev| 5.693058519106263| 10.36921449417913|14.56364295943342|
|    min|                10|             127.0|             25.0|
|    max|                97|             226.0|            214.0|
+-------+------------------+------------------+-----------------+



## 3. RDDs: transformaciones

Las transformaciones se aplican sobre `deportistaTotal` con operaciones propias de RDD (`filter`, `map`), sin pasar por DataFrames. Cada registro es una tupla con el orden de `COLUMNAS`: la edad está en la posición 3 y el género en la posición 2.

### 3.1 `MayorEdad`: deportistas de 18 años o más

Una edad igual a 0 o ausente corresponde a un dato no registrado; esos deportistas no se cuentan como mayores ni como menores de edad, y se informan por separado.

In [14]:
MayorEdad = deportistaTotal.filter(lambda r: r[3] is not None and r[3] >= 18)

edad_desconocida = deportistaTotal.filter(lambda r: r[3] is None or r[3] == 0)
menores = deportistaTotal.filter(lambda r: r[3] is not None and 0 < r[3] < 18)

n_mayores, n_menores, n_desconocida = MayorEdad.count(), menores.count(), edad_desconocida.count()

print(f"Mayores de edad (>= 18) : {n_mayores:,}")
print(f"Menores de edad (1-17)  : {n_menores:,}")
print(f"Edad no registrada      : {n_desconocida:,}")
print(f"Suma                    : {n_mayores + n_menores + n_desconocida:,} de {n_total:,}")
print("Edad mínima en MayorEdad:", MayorEdad.map(lambda r: r[3]).min())
print("Muestra:")
for registro in MayorEdad.take(5):
    print("   ", registro)

Mayores de edad (>= 18) : 123,244
Menores de edad (1-17)  : 5,959
Edad no registrada      : 6,368
Suma                    : 135,571 de 135,571
Edad mínima en MayorEdad: 18
Muestra:
    (1, 'A Dijiang', 1, 24, 180.0, 80.0, 199)
    (2, 'A Lamusi', 1, 23, 170.0, 60.0, 199)
    (3, 'Gunnar Nielsen Aaby', 1, 24, 0.0, 0.0, 273)
    (4, 'Edgar Lindenau Aabye', 1, 34, 0.0, 0.0, 278)
    (5, 'Christine Jacoba Aaftink', 2, 21, 185.0, 82.0, 705)


### 3.2 `Deportistas_mujer`: deportistas con género 2

Según la codificación de la fuente, el valor 2 de `genero` corresponde a mujer.

In [15]:
Deportistas_mujer = deportistaTotal.filter(lambda r: r[2] == 2)

n_mujeres = Deportistas_mujer.count()
n_hombres = deportistaTotal.filter(lambda r: r[2] == 1).count()

print(f"Mujeres (genero = 2)        : {n_mujeres:,}")
print(f"Hombres (genero = 1)        : {n_hombres:,}")
print(f"Otro código o ausente       : {n_total - n_mujeres - n_hombres:,}")
print("Valores de genero en el RDD :", sorted(Deportistas_mujer.map(lambda r: r[2]).distinct().collect()))
print("Muestra:")
for registro in Deportistas_mujer.take(5):
    print("   ", registro)

Mujeres (genero = 2)        : 33,981
Hombres (genero = 1)        : 101,590
Otro código o ausente       : 0


Valores de genero en el RDD : [2]
Muestra:
    (5, 'Christine Jacoba Aaftink', 2, 21, 185.0, 82.0, 705)
    (8, 'Cornelia Cor Aalten Strannood', 2, 18, 168.0, 0.0, 705)
    (13, 'Minna Maarit Aalto', 2, 30, 159.0, 55.5, 350)
    (14, 'Pirjo Hannele Aalto Mattila', 2, 32, 171.0, 65.0, 350)
    (21, 'Ragnhild Margrethe Aamodt', 2, 27, 163.0, 0.0, 742)


### 3.3 Conversión a mayúsculas

Se convierten a mayúsculas los campos de texto de `deportistaTotal`; en este conjunto el único campo textual es `nombre`. Los valores numéricos no se modifican.

Antes de reasignar el nombre se guarda una referencia al RDD original (`deportistaTotal_original`) para comparar ambos y mantener la trazabilidad. Como `map` conserva el orden y las particiones, los registros de ambos RDDs pueden emparejarse con `zip`.

In [16]:
deportistaTotal_original = deportistaTotal

deportistaTotal = deportistaTotal_original.map(
    lambda r: tuple(v.upper() if isinstance(v, str) else v for v in r)
).cache()

print(f"{'deportista_id':>13s}  {'nombre original':35s} nombre en mayúsculas")
for original, mayus in deportistaTotal_original.zip(deportistaTotal).take(10):
    print(f"{original[0]:>13}  {original[1]:35s} {mayus[1]}")

pares = deportistaTotal_original.zip(deportistaTotal)
numericos_alterados = pares.filter(lambda p: p[0][0] != p[1][0] or p[0][2:] != p[1][2:]).count()
sin_minusculas = deportistaTotal.filter(lambda r: r[1] != r[1].upper()).count()

print()
print(f"Registros en deportistaTotal       : {deportistaTotal.count():,}")
print(f"Nombres que aún tienen minúsculas  : {sin_minusculas:,}")
print(f"Registros con valores numéricos alterados: {numericos_alterados:,}")

deportista_id  nombre original                     nombre en mayúsculas
            1  A Dijiang                           A DIJIANG
            2  A Lamusi                            A LAMUSI
            3  Gunnar Nielsen Aaby                 GUNNAR NIELSEN AABY
            4  Edgar Lindenau Aabye                EDGAR LINDENAU AABYE
            5  Christine Jacoba Aaftink            CHRISTINE JACOBA AAFTINK
            6  Per Knut Aaland                     PER KNUT AALAND
            7  John Aalberg                        JOHN AALBERG
            8  Cornelia Cor Aalten Strannood       CORNELIA COR AALTEN STRANNOOD
            9  Antti Sami Aalto                    ANTTI SAMI AALTO
           10  Einar Ferdinand Einari Aalto        EINAR FERDINAND EINARI AALTO



Registros en deportistaTotal       : 135,571
Nombres que aún tienen minúsculas  : 0
Registros con valores numéricos alterados: 0


### 3.4 Resumen de los puntos 2 y 3

| Elemento | Resultado |
|---|---|
| RDD `deportista` | 67.786 registros de `deportista.csv` en seis particiones de entre 11.095 y 11.424 registros |
| RDD `deportista2` | 67.785 registros de `deportista2.csv` |
| RDD `deportistaTotal` | 135.571 registros (67.786 + 67.785), sin identificadores repetidos; nombres en mayúsculas |
| RDD `MayorEdad` | 123.244 deportistas de 18 años o más |
| RDD `Deportistas_mujer` | 33.981 deportistas con `genero = 2` |
| DataFrame `deportista` | 135.571 filas y siete columnas tipadas |

**Conclusiones**

- La unión conserva todos los registros de ambos archivos: 135.571 deportistas con identificadores únicos y continuos, que corresponden a los `deportista_id` usados en `resultados.csv`.
- Del total, 123.244 deportistas son mayores de edad, 5.959 tienen entre 10 y 17 años y 6.368 no tienen edad registrada. Estos últimos no se clasifican como menores.
- Las mujeres representan 33.981 registros (25,1 %) y los hombres 101.590 (74,9 %).
- La conversión a mayúsculas afectó solo a los nombres: ningún valor numérico cambió.
- En el DataFrame quedan sin dato 6.368 edades (4,7 %), 33.916 alturas (25,0 %) y 34.885 pesos (25,7 %). Entre los valores registrados, la edad promedio es de 24,4 años (rango de 10 a 97), la altura promedio de 176,3 cm y el peso promedio de 72,0 kg.
- La falta de altura o peso en cerca de un cuarto de los deportistas limitará el cálculo del IMC a los registros que tengan ambos valores.

## 4. DataFrames: creación e integración — Sebastián

Este bloque continúa con `deportista`, construido por Yerko. No vuelve a leer ni reemplaza su entrada.
Se usa **Resultado como tabla central**, con una fila por registro de participación. Se conservan
los resultados sin correspondencia mediante joins izquierdos; no se deduplican participaciones.

Los nombres reales de los archivos difieren de los ejemplos de la consigna:

| DataFrame | Archivo | Clave | Lectura |
|---|---|---|---|
| Evento | evento.csv | cod_evento → evento_id | CSV con encabezado, coma |
| Resultado | resultados.csv | resultado_id | CSV con encabezado, punto y coma |
| Equipos | equipo.csv | id → equipo_id | CSV con encabezado, coma |
| Juego | juegos.json | juego_id | arreglo JSON, multiline |

Se declaran esquemas explícitos y lectura `FAILFAST`: una conversión inválida detiene el flujo.
`NA` en medalla se conserva como código de ausencia de medalla, distinto de un dato nulo.

**Anomalía de Evento:** 280 filas encapsulan el registro CSV completo en un único campo. Se retira esa capa usando `csv.reader`, se validan los tres campos y se entrega CSV normalizado a Spark. Se informa el número reparado y no se descartan filas ni se modifica la fuente.

`#N/A` en `deporte_id` se interpreta como nulo: es un atributo ausente, no la clave de Evento. Se muestran los eventos afectados y se conservan en la integración.

En Resultado hay nueve `evento_id = #N/A`: se leen como nulos y se conservan mediante el join izquierdo. No confundir `#N/A` (dato faltante) con `NA` en medalla (sin medalla).


In [17]:
from pyspark import StorageLevel
from pyspark.sql import functions as F

# Algunas filas son un CSV completo encapsulado en un único campo CSV.
import csv
import io

def normalizar_evento(linea):
    campos = next(csv.reader([linea], strict=True))
    reparada = len(campos) == 1
    if reparada:
        campos = next(csv.reader([campos[0]], strict=True))
    if len(campos) != 3:
        raise ValueError(f"Evento con número de campos inválido: {linea!r}")
    salida = io.StringIO()
    csv.writer(salida, lineterminator="").writerow(campos)
    return salida.getvalue(), reparada

lineas_evento = sc.textFile(str(DIR_DATOS / "evento.csv"))
eventos_normalizados = lineas_evento.map(normalizar_evento)
print("Filas de Evento con doble encapsulado reparado:", eventos_normalizados.filter(lambda r: r[1]).count())
Evento = (spark.read.option("header", True).option("mode", "FAILFAST").option("escape", '\"')
          .schema("cod_evento INT, evento STRING, deporte_id INT")
          .option("nullValue", "#N/A")
          .csv(eventos_normalizados.map(lambda r: r[0]))
          .withColumnRenamed("cod_evento", "evento_id"))
Resultado = (spark.read.option("header", True).option("sep", ";").option("mode", "FAILFAST")
             .schema("resultado_id INT, medalla STRING, deportista_id INT, juego_id INT, evento_id INT")
             .option("nullValue", "#N/A")
             .csv(str(DIR_DATOS / "resultados.csv")))
Equipos = (spark.read.option("header", True).option("mode", "FAILFAST")
           .schema("id INT, equipo STRING, sigla STRING")
           .csv(str(DIR_DATOS / "equipo.csv"))
           .withColumnRenamed("id", "equipo_id"))
Juego_fuente = (spark.read.option("multiLine", True).option("mode", "FAILFAST")
                .schema("juego_id INT, ano STRING, temporada INT, ciudad STRING")
                .json(str(DIR_DATOS / "juegos.json")))
Juego_fuente.show(5, truncate=False)
print("Eventos sin deporte_id:", Evento.filter(F.col("deporte_id").isNull()).count())
Evento.filter(F.col("deporte_id").isNull()).show(truncate=False)


Filas de Evento con doble encapsulado reparado: 280


+--------+-----------+---------+------+
|juego_id|ano        |temporada|ciudad|
+--------+-----------+---------+------+
|1       |1896 Verano|1896     |Verano|
|2       |1900 Verano|1900     |Verano|
|3       |1904 Verano|1904     |Verano|
|4       |1906 Verano|1906     |Verano|
|5       |1908 Verano|1908     |Verano|
+--------+-----------+---------+------+
only showing top 5 rows


Eventos sin deporte_id: 2


+---------+------------------------------------------+----------+
|evento_id|evento                                    |deporte_id|
+---------+------------------------------------------+----------+
|703      |Sailing Mixed 20+ Ton                     |NULL      |
|743      |Motorboating Mixed B-Class (Under 60 Feet)|NULL      |
+---------+------------------------------------------+----------+



### 4.1 Corrección semántica de Juegos

La fuente trae `ano = "1896 Verano"`, `temporada = 1896` y `ciudad = "Verano"`.
Se comprueba la coherencia de esos valores en **todas** las filas antes de renombrarlos:
`ano` pasa a `edicion`, `temporada` a `anio`, y `ciudad` a `temporada`.
Se conservan además los tres campos originales con sufijo `_fuente` para trazabilidad.
No hay una ciudad sede utilizable en este archivo; no se inventa una.

In [18]:
condicion_juego = (
    F.col("ciudad").isin("Verano", "Invierno")
    & F.col("temporada").isNotNull()
    & (F.col("ano") == F.concat_ws(" ", F.col("temporada"), F.col("ciudad")))
)
assert Juego_fuente.filter(~F.coalesce(condicion_juego, F.lit(False))).count() == 0, "Revisar estructura de Juegos"
Juego = Juego_fuente.select(
    "juego_id", F.col("ano").alias("edicion"), F.col("temporada").alias("anio"),
    F.col("ciudad").alias("temporada"), F.col("ano").alias("ano_fuente"),
    F.col("temporada").alias("temporada_fuente"), F.col("ciudad").alias("ciudad_fuente")
)
conteos_fuentes = {}
for nombre_tabla, tabla in [("Evento", Evento), ("Resultado", Resultado), ("Equipos", Equipos), ("Juego", Juego)]:
    conteos_fuentes[nombre_tabla] = tabla.count()
    print(nombre_tabla, "— filas:", conteos_fuentes[nombre_tabla])
    tabla.printSchema()
    tabla.show(5, truncate=False)
Resultado.groupBy("medalla").count().orderBy("medalla").show()

Evento — filas: 765
root
 |-- evento_id: integer (nullable = true)
 |-- evento: string (nullable = true)
 |-- deporte_id: integer (nullable = true)

+---------+--------------------------------+----------+
|evento_id|evento                          |deporte_id|
+---------+--------------------------------+----------+
|1        |Basketball Men's Basketball     |1         |
|2        |Judo Men's Extra-Lightweight    |2         |
|3        |Football Men's Football         |3         |
|4        |Tug-Of-War Men's Tug-Of-War     |4         |
|5        |Speed Skating Women's 500 metres|5         |
+---------+--------------------------------+----------+
only showing top 5 rows


Resultado — filas: 271116
root
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- deportista_id: integer (nullable = true)
 |-- juego_id: integer (nullable = true)
 |-- evento_id: integer (nullable = true)

+------------+-------+-------------+--------+---------+
|resultado_id|medalla|deportista_id|juego_id|evento_id|
+------------+-------+-------------+--------+---------+
|1           |NA     |1            |39      |1        |
|2           |NA     |2            |49      |2        |
|3           |NA     |3            |7       |3        |
|4           |Gold   |4            |2       |4        |
|5           |NA     |5            |36      |5        |
+------------+-------+-------------+--------+---------+
only showing top 5 rows


Equipos — filas: 1184
root
 |-- equipo_id: integer (nullable = true)
 |-- equipo: string (nullable = true)
 |-- sigla: string (nullable = true)

+---------+---------------------+-----+
|equipo_id|equipo               |sigla|
+---------+---------------------+-----+
|1        |30. Februar          |AUT  |
|2        |A North American Team|MEX  |
|3        |Acipactli            |MEX  |
|4        |Acturus              |ARG  |
|5        |Afghanistan          |AFG  |
+---------+---------------------+-----+
only showing top 5 rows


Juego — filas: 51
root
 |-- juego_id: integer (nullable = true)
 |-- edicion: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- temporada: string (nullable = true)
 |-- ano_fuente: string (nullable = true)
 |-- temporada_fuente: integer (nullable = true)
 |-- ciudad_fuente: string (nullable = true)

+--------+-----------+----+---------+-----------+----------------+-------------+
|juego_id|edicion    |anio|temporada|ano_fuente |temporada_fuente|ciudad_fuente|
+--------+-----------+----+---------+-----------+----------------+-------------+
|1       |1896 Verano|1896|Verano   |1896 Verano|1896            |Verano       |
|2       |1900 Verano|1900|Verano   |1900 Verano|1900            |Verano       |
|3       |1904 Verano|1904|Verano   |1904 Verano|1904            |Verano       |
|4       |1906 Verano|1906|Verano   |1906 Verano|1906            |Verano       |
|5       |1908 Verano|1908|Verano   |1908 Verano|1908            |Verano       |
+--------+-----------+----+-------

+-------+------+
|medalla| count|
+-------+------+
| Bronze| 13295|
|   Gold| 13372|
|     NA|231333|
| Silver| 13116|
+-------+------+



### 4.2 Controles de claves y correspondencias

Una clave repetida en una dimensión multiplicaría resultados. Se detiene la integración si
hay claves nulas o duplicadas, incluido `resultado_id`. Los joins anti permiten contar las
referencias sin correspondencia antes de unir. Un equipo ausente se conserva como nulo:
no se asigna a un país inventado ni se elimina la participación.

In [19]:
def verificar_clave(tabla, clave, nombre_tabla):
    resumen = tabla.agg(F.count("*").alias("filas"), F.count(clave).alias("presentes"),
                       F.countDistinct(clave).alias("distintas")).first()
    print(nombre_tabla, clave, resumen.asDict())
    assert resumen.filas == resumen.presentes == resumen.distintas, f"Clave no válida en {nombre_tabla}"

for tabla, clave, nombre_tabla in [
    (deportista, "deportista_id", "deportista"), (Evento, "evento_id", "Evento"),
    (Equipos, "equipo_id", "Equipos"), (Juego, "juego_id", "Juego"),
    (Resultado, "resultado_id", "Resultado")
]:
    verificar_clave(tabla, clave, nombre_tabla)

for nombre_control, izquierda, derecha, clave in [
    ("Resultados sin deportista", Resultado, deportista, "deportista_id"),
    ("Resultados sin evento", Resultado, Evento, "evento_id"),
    ("Resultados sin juego", Resultado, Juego, "juego_id"),
    ("Deportistas sin equipo", deportista, Equipos, "equipo_id")
]:
    sin_correspondencia = izquierda.join(derecha.select(clave), clave, "left_anti")
    print(nombre_control, ":", sin_correspondencia.count())
    sin_correspondencia.select(clave).distinct().orderBy(clave).show(5)

deportista deportista_id {'filas': 135571, 'presentes': 135571, 'distintas': 135571}


Evento evento_id {'filas': 765, 'presentes': 765, 'distintas': 765}


Equipos equipo_id {'filas': 1184, 'presentes': 1184, 'distintas': 1184}


Juego juego_id {'filas': 51, 'presentes': 51, 'distintas': 51}


Resultado resultado_id {'filas': 271116, 'presentes': 271116, 'distintas': 271116}


Resultados sin deportista : 0


... (3 outputs left out)

Resultados sin juego : 0


+--------+
|juego_id|
+--------+
+--------+



Deportistas sin equipo : 143


+---------+
|equipo_id|
+---------+
|        0|
+---------+



### 4.3 Integración y optimización

Se seleccionan únicamente las columnas necesarias de cada entrada (más los campos de
trazabilidad de Juegos). Los esquemas explícitos evitan inferencia y mantienen claves compatibles.
Las dimensiones Evento, Equipos y Juego son archivos pequeños (menos de 100 KB cada uno);
se verifica también que cada dimensión tenga menos de 5.000 filas antes de usar `broadcast`.
El plan físico permite observar los joins. No se afirma una mejora de tiempo sin una comparación medida.

El resultado se persiste con `MEMORY_AND_DISK` porque se reutiliza en la inspección y en las
consultas posteriores de José. Se materializa con `count`. Se debe liberar con
`df_integrado.unpersist()` **después** de terminar esas consultas.

In [20]:
for nombre_tabla, archivo in [("Evento", "evento.csv"), ("Equipos", "equipo.csv"), ("Juego", "juegos.json")]:
    tamano = (DIR_DATOS / archivo).stat().st_size
    print(nombre_tabla, "— bytes de fuente:", tamano, "— filas:", conteos_fuentes[nombre_tabla])
    assert tamano < 100_000 and conteos_fuentes[nombre_tabla] < 5_000, "Reevaluar uso de broadcast"

df_union = (Resultado.select("resultado_id", "medalla", "deportista_id", "juego_id", "evento_id")
    .join(deportista.select("deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id"), "deportista_id", "left")
    .join(F.broadcast(Evento.select("evento_id", "evento", "deporte_id")), "evento_id", "left")
    .join(F.broadcast(Equipos.select("equipo_id", "equipo", "sigla")), "equipo_id", "left")
    .join(F.broadcast(Juego), "juego_id", "left"))
df_union.explain(mode="simple")

Evento — bytes de fuente: 35144 — filas: 765
Equipos — bytes de fuente: 24791 — filas: 1184
Juego — bytes de fuente: 5539 — filas: 51


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [juego_id#322, equipo_id#15, evento_id#323, deportista_id#321, resultado_id#319, medalla#320, nombre#10, genero#11, edad#12, altura#13, peso#14, evento#316, deporte_id#317, equipo#325, sigla#326, edicion#379, anio#380, temporada#381, ano_fuente#382, temporada_fuente#383, ciudad_fuente#384]
   +- BroadcastHashJoin [juego_id#322], [juego_id#329], LeftOuter, BuildRight, false, false
      :- Project [equipo_id#15, evento_id#323, deportista_id#321, resultado_id#319, medalla#320, juego_id#322, nombre#10, genero#11, edad#12, altura#13, peso#14, evento#316, deporte_id#317, equipo#325, sigla#326]
      :  +- BroadcastHashJoin [equipo_id#15], [equipo_id#328], LeftOuter, BuildRight, false, false
      :     :- Project [evento_id#323, deportista_id#321, resultado_id#319, medalla#320, juego_id#322, nombre#10, genero#11, edad#12, altura#13, peso#14, equipo_id#15, evento#316, deporte_id#317]
      :     :  +- BroadcastHashJoin [event

## 5. Paralelismo — Sebastián

`repartition(5)` redistribuye los registros mediante shuffle. Las cinco particiones son
particiones lógicas, no cinco máquinas; no implican por sí solas mayor velocidad local.
Se verifica la distribución después de materializar el DataFrame persistido.

In [21]:
n_particiones_antes = df_union.rdd.getNumPartitions()
df_integrado = df_union.repartition(5).persist(StorageLevel.MEMORY_AND_DISK)
n_integrado = df_integrado.count()
n_particiones = df_integrado.rdd.getNumPartitions()
filas_por_particion = df_integrado.rdd.mapPartitions(lambda filas: [sum(1 for _ in filas)]).collect()
print("Filas de Resultado:", conteos_fuentes["Resultado"])
print("Filas integradas:", n_integrado)
print("Particiones antes de repartition:", n_particiones_antes)
print("Particiones después de repartition(5):", n_particiones)
print("Filas por partición:", filas_por_particion)
print("Persistencia:", df_integrado.storageLevel)
assert n_particiones == 5
assert sum(filas_por_particion) == n_integrado == conteos_fuentes["Resultado"]
assert df_integrado.select("resultado_id").distinct().count() == n_integrado, "Se multiplicaron resultados"

Filas de Resultado: 271116
Filas integradas: 271116
Particiones antes de repartition: 8
Particiones después de repartition(5): 5
Filas por partición: [54224, 54224, 54223, 54221, 54224]
Persistencia: Disk Memory Serialized 1x Replicated


## 6. Inspección del DataFrame y entrega a José — Sebastián

`df_integrado` queda disponible para IMC, `Descripción_sexo` y agregaciones SQL.
`genero` mantiene 1 = hombre, 2 = mujer; `edad` está en años, `altura` en centímetros y
`peso` en kg según el contrato del bloque anterior. Los ceros de esas tres medidas ya fueron
convertidos a nulo por Yerko. `temporada` contiene Verano/Invierno y `anio` contiene el año.
`medalla` conserva Gold, Silver, Bronze y NA (sin medalla).

La unidad del análisis es la participación: una persona puede aparecer varias veces.
Los conteos de medallas por registro no equivalen necesariamente al medallero oficial de
pruebas por equipos. Se conserva `equipo_id` incluso cuando no existe en Equipos;
José deberá mostrar explícitamente esos casos si agrupa por nombre de equipo.

In [22]:
print("Número de filas:", n_integrado)
print("Tipo del objeto:", type(df_integrado))
print("Tipos de las columnas:", df_integrado.dtypes)
df_integrado.printSchema()
df_integrado.orderBy("resultado_id").show(10, truncate=False)
df_integrado.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["nombre", "evento", "equipo", "temporada", "edad", "altura", "peso"]
]).show()
print("Participaciones sin equipo identificado:", df_integrado.filter(F.col("equipo").isNull()).count())
df_integrado.filter(F.col("equipo").isNull()).select(
    "resultado_id", "deportista_id", "equipo_id", "equipo"
).orderBy("resultado_id").show(5)

Número de filas: 271116
Tipo del objeto: <class 'pyspark.sql.classic.dataframe.DataFrame'>
Tipos de las columnas: [('juego_id', 'int'), ('equipo_id', 'int'), ('evento_id', 'int'), ('deportista_id', 'int'), ('resultado_id', 'int'), ('medalla', 'string'), ('nombre', 'string'), ('genero', 'int'), ('edad', 'int'), ('altura', 'double'), ('peso', 'double'), ('evento', 'string'), ('deporte_id', 'int'), ('equipo', 'string'), ('sigla', 'string'), ('edicion', 'string'), ('anio', 'int'), ('temporada', 'string'), ('ano_fuente', 'string'), ('temporada_fuente', 'int'), ('ciudad_fuente', 'string')]
root
 |-- juego_id: integer (nullable = true)
 |-- equipo_id: integer (nullable = true)
 |-- evento_id: integer (nullable = true)
 |-- deportista_id: integer (nullable = true)
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = tru

+--------+---------+---------+-------------+------------+-------+------------------------+------+----+------+----+---------------------------------+----------+--------------+-----+-------------+----+---------+-------------+----------------+-------------+
|juego_id|equipo_id|evento_id|deportista_id|resultado_id|medalla|nombre                  |genero|edad|altura|peso|evento                           |deporte_id|equipo        |sigla|edicion      |anio|temporada|ano_fuente   |temporada_fuente|ciudad_fuente|
+--------+---------+---------+-------------+------------+-------+------------------------+------+----+------+----+---------------------------------+----------+--------------+-----+-------------+----+---------+-------------+----------------+-------------+
|39      |199      |1        |1            |1           |NA     |A Dijiang               |1     |24  |180.0 |80.0|Basketball Men's Basketball      |1         |China         |CHN  |1992 Verano  |1992|Verano   |1992 Verano  |1992        

+------+------+------+---------+----+------+-----+
|nombre|evento|equipo|temporada|edad|altura| peso|
+------+------+------+---------+----+------+-----+
|     0|     9|   200|        0|9474| 60171|62875|
+------+------+------+---------+----+------+-----+

Participaciones sin equipo identificado: 200


+------------+-------------+---------+------+
|resultado_id|deportista_id|equipo_id|equipo|
+------------+-------------+---------+------+
|         654|          378|        0|  NULL|
|        5946|         3341|        0|  NULL|
|        6147|         3459|        0|  NULL|
|        6148|         3459|        0|  NULL|
|        6149|         3459|        0|  NULL|
+------------+-------------+---------+------+
only showing top 5 rows


### Cierre del bloque de Sebastián

Los controles anteriores verifican que cada resultado sigue apareciendo exactamente una vez,
que no se pierden referencias sin equipo y que la tabla tiene cinco particiones.
La anomalía semántica de Juegos queda corregida y trazable sin modificar el archivo original.
Las celdas siguientes corresponden al bloque de José: columnas calculadas y cuatro agregaciones SQL.

## 7. Columnas calculadas — José

Este bloque continúa desde `df_integrado`, en la misma sesión. No vuelve a leer archivos ni reconstruye los RDDs.
La unidad de análisis es la **participación**: 271.116 filas de Resultado, donde una misma persona puede aparecer varias veces.

**Entorno de esta ejecución.** El notebook completo se ejecutó en Linux (contenedor con Java 17 Temurin, Python 3.14 y PySpark 4.3.0 instalados desde `uv.lock`), igual que la ejecución de Sebastián. En Windows nativo, PySpark 4.3 falla de forma intermitente en `take` sobre RDDs grandes (*Python worker exited unexpectedly*), lo que afecta a la sección 2.2; las versiones efectivas se imprimen en la celda 1.2.

### 7.1 Comprobación de la entrada

Antes de calcular se verifica que existan las columnas requeridas con sus tipos, que `resultado_id` sea único y qué códigos traen `medalla` y `genero`. El conteo inicial queda en `n_entrada` para compararlo después.

In [23]:
columnas_requeridas = {
    "resultado_id": "int", "deportista_id": "int", "equipo_id": "int", "equipo": "string",
    "sigla": "string", "genero": "int", "edad": "int", "altura": "double", "peso": "double",
    "medalla": "string", "temporada": "string", "anio": "int",
}
tipos_entrada = dict(df_integrado.dtypes)
faltantes = {c: t for c, t in columnas_requeridas.items() if tipos_entrada.get(c) != t}
assert not faltantes, f"Columnas ausentes o con otro tipo: {faltantes}"

n_entrada = df_integrado.count()
columnas_entrada = df_integrado.columns
resumen_entrada = df_integrado.agg(
    F.countDistinct("resultado_id").alias("resultado_id_distintos"),
    F.countDistinct("deportista_id").alias("deportistas_distintos"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}") for c in ["equipo", "genero", "edad", "altura", "peso", "medalla", "temporada"]],
).first()

print(f"Participaciones de entrada : {n_entrada:,}")
print(f"Columnas de entrada        : {len(columnas_entrada)}")
for clave, valor in resumen_entrada.asDict().items():
    print(f"  {clave:24s}: {valor:,}")
assert resumen_entrada.resultado_id_distintos == n_entrada, "resultado_id no es único"

print("Códigos de medalla:")
df_integrado.groupBy("medalla").count().orderBy("medalla").show()
print("Códigos de genero:")
df_integrado.groupBy("genero").count().orderBy("genero").show()
print("Valores de temporada:")
df_integrado.groupBy("temporada").count().orderBy("temporada").show()

Participaciones de entrada : 271,116
Columnas de entrada        : 21
  resultado_id_distintos  : 271,116
  deportistas_distintos   : 135,571
  nulos_equipo            : 200
  nulos_genero            : 0
  nulos_edad              : 9,474
  nulos_altura            : 60,171
  nulos_peso              : 62,875
  nulos_medalla           : 0
  nulos_temporada         : 0
Códigos de medalla:


+-------+------+
|medalla| count|
+-------+------+
| Bronze| 13295|
|   Gold| 13372|
|     NA|231333|
| Silver| 13116|
+-------+------+

Códigos de genero:


+------+------+
|genero| count|
+------+------+
|     1|196594|
|     2| 74522|
+------+------+

Valores de temporada:
+---------+------+
|temporada| count|
+---------+------+
| Invierno| 48564|
|   Verano|222552|
+---------+------+



### 7.2 `IMC` y `Descripción_sexo`

- **IMC** = peso (kg) / altura (m)². La altura viene en centímetros, por lo que se divide por 100 antes de elevarla al cuadrado.
- Se calcula **solo** cuando altura y peso están presentes, son positivos y finitos (se descartan nulos, ceros, negativos, NaN e infinitos). En otro caso el IMC queda **nulo**: no se imputa ningún valor. Así no hay divisiones por cero ni IMC engañosos.
- El IMC se guarda con toda su precisión; solo se redondea al mostrarlo.
- **Descripción_sexo**: 1 = `Hombre`, 2 = `Mujer`; cualquier otro código o nulo = `No informado`.

Las reglas se definen una sola vez en `agregar_columnas_calculadas`, para aplicar exactamente el mismo código a los datos oficiales y a los casos de control.

In [24]:
def es_medida_valida(columna):
    """Presente, numérica, finita y positiva."""
    return (columna.isNotNull() & ~F.isnan(columna)
            & (columna > 0) & (columna < float("inf")))


def agregar_columnas_calculadas(df):
    altura_m = F.col("altura") / 100.0
    imc_valido = es_medida_valida(F.col("altura")) & es_medida_valida(F.col("peso"))
    return (df
            .withColumn("IMC", F.when(imc_valido, F.col("peso") / (altura_m * altura_m)))
            .withColumn("Descripción_sexo",
                        F.when(F.col("genero") == 1, "Hombre")
                         .when(F.col("genero") == 2, "Mujer")
                         .otherwise("No informado")))


df_analisis = agregar_columnas_calculadas(df_integrado)

n_analisis = df_analisis.count()
print(f"Filas df_integrado: {n_entrada:,} | filas df_analisis: {n_analisis:,}")
print("Columnas agregadas:", [c for c in df_analisis.columns if c not in columnas_entrada])
assert n_analisis == n_entrada, "Se modificó el número de filas"
assert df_analisis.columns == columnas_entrada + ["IMC", "Descripción_sexo"], "No se conservaron las columnas de entrada"
assert df_analisis.select("resultado_id").distinct().count() == n_analisis, "resultado_id dejó de ser único"

print("Muestra (IMC redondeado solo para mostrar):")
(df_analisis
 .select("resultado_id", "altura", "peso", F.round("IMC", 2).alias("IMC"), "genero", "Descripción_sexo")
 .orderBy("resultado_id")
 .show(10))

Filas df_integrado: 271,116 | filas df_analisis: 271,116
Columnas agregadas: ['IMC', 'Descripción_sexo']


Muestra (IMC redondeado solo para mostrar):


+------------+------+----+-----+------+----------------+
|resultado_id|altura|peso|  IMC|genero|Descripción_sexo|
+------------+------+----+-----+------+----------------+
|           1| 180.0|80.0|24.69|     1|          Hombre|
|           2| 170.0|60.0|20.76|     1|          Hombre|
|           3|  NULL|NULL| NULL|     1|          Hombre|
|           4|  NULL|NULL| NULL|     1|          Hombre|
|           5| 185.0|82.0|23.96|     2|           Mujer|
|           6| 185.0|82.0|23.96|     2|           Mujer|
|           7| 185.0|82.0|23.96|     2|           Mujer|
|           8| 185.0|82.0|23.96|     2|           Mujer|
|           9| 185.0|82.0|23.96|     2|           Mujer|
|          10| 185.0|82.0|23.96|     2|           Mujer|
+------------+------+----+-----+------+----------------+
only showing top 10 rows


In [25]:
cobertura_imc = df_analisis.agg(
    F.count("*").alias("participaciones"),
    F.count("IMC").alias("imc_valido"),
    F.sum(F.col("IMC").isNull().cast("int")).alias("imc_nulo"),
    F.sum((F.col("altura").isNull() | F.col("peso").isNull()).cast("int")).alias("sin_altura_o_peso"),
).first()
print(f"IMC válido : {cobertura_imc.imc_valido:,} ({cobertura_imc.imc_valido / n_analisis:.1%})")
print(f"IMC nulo   : {cobertura_imc.imc_nulo:,} ({cobertura_imc.imc_nulo / n_analisis:.1%})")
print(f"  de ellos, sin altura o sin peso: {cobertura_imc.sin_altura_o_peso:,}")
assert cobertura_imc.imc_valido + cobertura_imc.imc_nulo == n_analisis

print("Resumen del IMC válido:")
df_analisis.select("IMC").summary("count", "mean", "min", "25%", "50%", "75%", "max").show()

print("Distribución de Descripción_sexo:")
df_analisis.groupBy("genero", "Descripción_sexo").count().orderBy("genero").show()

IMC válido : 206,853 (76.3%)
IMC nulo   : 64,263 (23.7%)
  de ellos, sin altura o sin peso: 64,263
Resumen del IMC válido:


+-------+------------------+
|summary|               IMC|
+-------+------------------+
|  count|            206853|
|   mean|22.784519726470922|
|    min| 8.360954343217175|
|    25%|20.957171162932475|
|    50%|22.530864197530864|
|    75%|24.212293388429753|
|    max| 63.90157962315983|
+-------+------------------+

Distribución de Descripción_sexo:


+------+----------------+------+
|genero|Descripción_sexo| count|
+------+----------------+------+
|     1|          Hombre|196594|
|     2|           Mujer| 74522|
+------+----------------+------+



**Interpretación — columnas calculadas**

- `df_analisis` conserva las 271.116 participaciones y las 21 columnas de entrada, y agrega `IMC` y `Descripción_sexo` (23 columnas).
- El IMC se pudo calcular en 206.853 participaciones (76,3 %). Las 64.263 restantes (23,7 %) quedan nulas, y en todas falta la altura o el peso. No aparecieron valores no positivos ni no finitos en los datos oficiales: los ceros ya se habían convertido a nulo en la sección 2.7.
- Entre los IMC válidos, la mediana es 22,53 y el promedio 22,78; la mitad central está entre 20,96 y 24,21. Los extremos (8,36 y 63,90) son poco plausibles para deportistas y podrían ser errores de registro de la fuente. Se informan, pero no se corrigen ni se eliminan, porque la consigna no define un criterio para hacerlo.
- `genero` solo trae los códigos 1 y 2: 196.594 participaciones de hombres y 74.522 de mujeres. No hay registros `No informado`.

### 7.3 Casos de control del IMC

Filas artificiales, **separadas de los datos oficiales** (no se agregan a `df_analisis`), para comprobar la fórmula y el tratamiento de valores inválidos. El caso conocido de 180 cm y 80 kg debe dar aproximadamente 24,69.

In [26]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

esquema_control = StructType([
    StructField("caso", StringType()), StructField("genero", IntegerType()),
    StructField("altura", DoubleType()), StructField("peso", DoubleType()),
])
casos_control = spark.createDataFrame([
    ("conocido 180 cm / 80 kg", 1, 180.0, 80.0),
    ("altura nula", 2, None, 70.0),
    ("peso nulo", 1, 170.0, None),
    ("altura cero", 2, 0.0, 70.0),
    ("peso negativo", 1, 170.0, -5.0),
    ("altura NaN", 2, float("nan"), 70.0),
    ("peso infinito", 1, 170.0, float("inf")),
    ("genero 3 (no informado)", 3, 160.0, 55.0),
    ("genero nulo", None, 165.0, 60.0),
], esquema_control)

control = agregar_columnas_calculadas(casos_control)
control.select("caso", "altura", "peso", F.round("IMC", 4).alias("IMC"), "genero", "Descripción_sexo").show(truncate=False)

resultado_control = {r.caso: r for r in control.collect()}
assert abs(resultado_control["conocido 180 cm / 80 kg"].IMC - 80 / 1.8 ** 2) < 1e-9
assert round(resultado_control["conocido 180 cm / 80 kg"].IMC, 2) == 24.69
assert all(resultado_control[c].IMC is None for c in
           ["altura nula", "peso nulo", "altura cero", "peso negativo", "altura NaN", "peso infinito"])
assert resultado_control["genero 3 (no informado)"].Descripción_sexo == "No informado"
assert resultado_control["genero nulo"].Descripción_sexo == "No informado"
print("Casos de control: OK")

+-----------------------+------+--------+-------+------+----------------+
|caso                   |altura|peso    |IMC    |genero|Descripción_sexo|
+-----------------------+------+--------+-------+------+----------------+
|conocido 180 cm / 80 kg|180.0 |80.0    |24.6914|1     |Hombre          |
|altura nula            |NULL  |70.0    |NULL   |2     |Mujer           |
|peso nulo              |170.0 |NULL    |NULL   |1     |Hombre          |
|altura cero            |0.0   |70.0    |NULL   |2     |Mujer           |
|peso negativo          |170.0 |-5.0    |NULL   |1     |Hombre          |
|altura NaN             |NaN   |70.0    |NULL   |2     |Mujer           |
|peso infinito          |170.0 |Infinity|NULL   |1     |Hombre          |
|genero 3 (no informado)|160.0 |55.0    |21.4844|3     |No informado    |
|genero nulo            |165.0 |60.0    |22.0386|NULL  |No informado    |
+-----------------------+------+--------+-------+------+----------------+

Casos de control: OK


## 8. Agregaciones con Spark SQL — José

`df_analisis` se registra como la vista temporal `participaciones_olimpicas` y las cuatro agregaciones se ejecutan con `spark.sql`.

Criterios comunes:

- `COUNT(*)` cuenta participaciones; `COUNT(edad)` o `COUNT(altura)` cuenta las observaciones realmente usadas por cada estadística. `SUM`, `AVG`, `MAX` y `MIN` omiten los nulos; **no se reemplazan por cero**.
- `NA` en `medalla` significa sin medalla y no se cuenta como medalla.
- Antes de clasificar se comprueba que `medalla` no traiga otros códigos.

In [27]:
df_analisis.createOrReplaceTempView("participaciones_olimpicas")

codigos_medalla = {r.medalla for r in spark.sql(
    "SELECT DISTINCT medalla FROM participaciones_olimpicas").collect()}
print("Códigos de medalla presentes:", sorted(codigos_medalla, key=str))
assert codigos_medalla <= {"Gold", "Silver", "Bronze", "NA"}, f"Códigos no previstos: {codigos_medalla}"

medallas_entrada = {r.medalla: r.n for r in spark.sql("""
    SELECT medalla, COUNT(*) AS n FROM participaciones_olimpicas GROUP BY medalla
""").collect()}
print("Conteo global por código:", medallas_entrada)

Códigos de medalla presentes: ['Bronze', 'Gold', 'NA', 'Silver']
Conteo global por código: {'NA': 231333, 'Silver': 13116, 'Gold': 13372, 'Bronze': 13295}


### 8.1 Medallas por equipo

Agregación condicional sobre todas las participaciones: cada fila suma 1 en la columna de su medalla. Por eso aparecen también los equipos con participaciones pero sin medallas, con ceros.

- Se agrupa por `equipo_id` **y** nombre, para no fusionar equipos distintos que comparten nombre.
- Las participaciones sin equipo identificado se muestran como `Equipo no identificado`, conservando su `equipo_id` original.
- El alcance son los equipos con al menos una participación; los equipos del catálogo sin participaciones no se agregan.
- `total_medallas` es una columna de apoyo para ordenar y revisar.

In [28]:
medallas_equipo = spark.sql("""
    SELECT equipo_id,
           COALESCE(equipo, 'Equipo no identificado') AS equipo,
           oro, plata, bronce,
           oro + plata + bronce AS total_medallas
    FROM (
        SELECT equipo_id,
               equipo,
               SUM(CASE WHEN medalla = 'Gold'   THEN 1 ELSE 0 END) AS oro,
               SUM(CASE WHEN medalla = 'Silver' THEN 1 ELSE 0 END) AS plata,
               SUM(CASE WHEN medalla = 'Bronze' THEN 1 ELSE 0 END) AS bronce
        FROM participaciones_olimpicas
        GROUP BY equipo_id, equipo
    ) AS por_equipo
    ORDER BY total_medallas DESC, oro DESC, plata DESC, equipo
""")

print("Primeros 15 equipos por registros de medalla:")
medallas_equipo.show(15, truncate=False)

print("Participaciones sin equipo identificado:")
medallas_equipo.filter(F.col("equipo") == "Equipo no identificado").show(truncate=False)

Primeros 15 equipos por registros de medalla:


+---------+-------------+----+-----+------+--------------+
|equipo_id|equipo       |oro |plata|bronce|total_medallas|
+---------+-------------+----+-----+------+--------------+
|1096     |United States|2515|1546 |1249  |5310          |
|978      |Soviet Union |1110|766  |728   |2604          |
|399      |Germany      |636 |603  |651   |1890          |
|413      |Great Britain|531 |591  |578   |1700          |
|362      |France       |452 |524  |582   |1558          |
|507      |Italy        |536 |507  |487   |1530          |
|1012     |Sweden       |450 |479  |510   |1439          |
|66       |Australia    |343 |453  |506   |1302          |
|174      |Canada       |422 |407  |407   |1236          |
|477      |Hungary      |429 |329  |363   |1121          |
|302      |East Germany |409 |325  |283   |1017          |
|705      |Netherlands  |275 |323  |395   |993           |
|874      |Russia       |309 |291  |361   |961           |
|199      |China        |323 |332  |273   |928          

+---------+----------------------+---+-----+------+--------------+
|equipo_id|equipo                |oro|plata|bronce|total_medallas|
+---------+----------------------+---+-----+------+--------------+
|0        |Equipo no identificado|0  |0    |0     |0             |
+---------+----------------------+---+-----+------+--------------+



In [29]:
conciliacion = medallas_equipo.agg(
    F.count("*").alias("equipos"),
    F.sum(F.when(F.col("total_medallas") == 0, 1).otherwise(0)).alias("equipos_sin_medallas"),
    F.sum("oro").alias("oro"), F.sum("plata").alias("plata"), F.sum("bronce").alias("bronce"),
).first()

print(f"Equipos con participaciones: {conciliacion.equipos:,} (sin medallas: {conciliacion.equipos_sin_medallas:,})")
print(f"{'medalla':8s} {'por equipos':>12s} {'entrada':>9s}")
for col, codigo in [("oro", "Gold"), ("plata", "Silver"), ("bronce", "Bronze")]:
    print(f"{col:8s} {conciliacion[col]:12,} {medallas_entrada[codigo]:9,}")
    assert conciliacion[col] == medallas_entrada[codigo], f"No concilia {col}"
print("Conciliación de medallas: OK")

Equipos con participaciones: 1,012 (sin medallas: 507)
medalla   por equipos   entrada
oro            13,372    13,372
plata          13,116    13,116
bronce         13,295    13,295
Conciliación de medallas: OK


**Interpretación — medallas por equipo**

- Los registros de medalla de todos los equipos suman exactamente los de la entrada: 13.372 de oro, 13.116 de plata y 13.295 de bronce (39.783 en total). No se perdió ni se duplicó ninguna medalla.
- Hay 1.012 equipos con al menos una participación. 507 de ellos no registran medallas y aparecen con ceros en las tres columnas.
- Las 200 participaciones sin equipo identificado (`equipo_id = 0`) forman un grupo propio y no registran medallas, así que excluirlas no habría cambiado los totales. Se conservan igual para que la conciliación sea completa.
- Encabezan la lista United States (5.310 registros; 2.515 de oro), Soviet Union (2.604) y Germany (1.890). Las entidades históricas como Soviet Union, East Germany y Russia figuran como equipos distintos, porque así vienen en la fuente.
- Son **registros de medalla por participación**, no un medallero oficial: en una prueba colectiva cada integrante del equipo suma una medalla. Por eso el conteo favorece a los países con más deportes de equipo y más participantes.

### 8.2 Edad por tipo de medalla

Suma, promedio, máximo y mínimo de `edad` por medalla. `NA` se mantiene como grupo **complementario** `Sin medalla` para la trazabilidad, separado de Oro, Plata y Bronce. `n_participaciones` y `n_edad_valida` documentan cuántas filas tiene cada grupo y cuántas edades entran en el cálculo.

La edad proviene de la tabla Deportista y se repite en todas las participaciones de una persona; sin más información no puede afirmarse que sea su edad exacta en cada edición.

In [30]:
edad_medalla = spark.sql("""
    SELECT CASE medalla WHEN 'Gold' THEN 'Oro' WHEN 'Silver' THEN 'Plata'
                        WHEN 'Bronze' THEN 'Bronce' WHEN 'NA' THEN 'Sin medalla' END AS tipo_medalla,
           medalla                    AS codigo_medalla,
           COUNT(*)                   AS n_participaciones,
           COUNT(edad)                AS n_edad_valida,
           SUM(edad)                  AS suma_edad,
           ROUND(AVG(edad), 2)        AS promedio_edad,
           MAX(edad)                  AS maximo_edad,
           MIN(edad)                  AS minimo_edad
    FROM participaciones_olimpicas
    GROUP BY medalla
    ORDER BY CASE medalla WHEN 'Gold' THEN 1 WHEN 'Silver' THEN 2
                          WHEN 'Bronze' THEN 3 ELSE 4 END
""")
edad_medalla.show(truncate=False)


def verificar_agregacion(tabla, promedio, minimo, maximo):
    """Concilia participaciones con la entrada y comprueba mínimo <= promedio <= máximo."""
    filas = tabla.collect()
    total = sum(r.n_participaciones for r in filas)
    assert total == n_entrada, f"Las participaciones suman {total:,}, no {n_entrada:,}"
    for r in filas:
        if r[promedio] is not None:
            assert r[minimo] <= r[promedio] <= r[maximo], f"Promedio fuera de rango: {r}"
    print(f"Participaciones conciliadas: {total:,} = {n_entrada:,} | mínimo <= promedio <= máximo: OK")


verificar_agregacion(edad_medalla, "promedio_edad", "minimo_edad", "maximo_edad")

+------------+--------------+-----------------+-------------+---------+-------------+-----------+-----------+
|tipo_medalla|codigo_medalla|n_participaciones|n_edad_valida|suma_edad|promedio_edad|maximo_edad|minimo_edad|
+------------+--------------+-----------------+-------------+---------+-------------+-----------+-----------+
|Oro         |Gold          |13372            |13224        |313374   |23.7         |64         |11         |
|Plata       |Silver        |13116            |12821        |304518   |23.75        |73         |11         |
|Bronce      |Bronze        |13295            |13006        |308071   |23.69        |72         |10         |
|Sin medalla |NA            |231333           |222591       |5275565  |23.7         |97         |11         |
+------------+--------------+-----------------+-------------+---------+-------------+-----------+-----------+



Participaciones conciliadas: 271,116 = 271,116 | mínimo <= promedio <= máximo: OK


**Interpretación — edad por tipo de medalla**

- La edad promedio es prácticamente igual en todos los grupos: 23,70 años en oro, 23,75 en plata, 23,69 en bronce y 23,70 sin medalla. En estos datos la edad no distingue a quienes obtienen medalla.
- Los mínimos van de 10 a 11 años. El máximo es mayor entre quienes no obtuvieron medalla (97 años) que entre los medallistas (64 a 73 años).
- Cobertura de la edad: 13.224 de 13.372 participaciones con oro (98,9 %), 12.821 de 13.116 con plata (97,8 %), 13.006 de 13.295 con bronce (97,8 %) y 222.591 de 231.333 sin medalla (96,2 %). La suma y el promedio se calculan solo sobre esas edades válidas.
- El promedio es por participación: quien participa varias veces pesa varias veces. Además, la misma edad del deportista se repite en todas sus participaciones.

### 8.3 Altura por temporada (`temporada`)

Suma, promedio, máximo y mínimo de `altura` por temporada, en **centímetros**. Se usa la columna `temporada` ya corregida por Sebastián (Verano / Invierno). Los nulos se omiten y se cuantifican con `n_altura_valida`.

In [31]:
temporada = spark.sql("""
    SELECT temporada,
           COUNT(*)                 AS n_participaciones,
           COUNT(altura)            AS n_altura_valida,
           CAST(SUM(altura) AS DECIMAL(14, 1)) AS suma_altura_cm,
           ROUND(AVG(altura), 2)    AS promedio_altura_cm,
           MAX(altura)              AS maximo_altura_cm,
           MIN(altura)              AS minimo_altura_cm
    FROM participaciones_olimpicas
    GROUP BY temporada
    ORDER BY temporada DESC
""")
temporada.show(truncate=False)

grupos_temporada = {r.temporada for r in temporada.collect()}
print("Grupos de temporada:", grupos_temporada,
      "" if grupos_temporada == {"Verano", "Invierno"} else "<- revisar grupos inesperados")
verificar_agregacion(temporada, "promedio_altura_cm", "minimo_altura_cm", "maximo_altura_cm")

+---------+-----------------+---------------+--------------+------------------+----------------+----------------+
|temporada|n_participaciones|n_altura_valida|suma_altura_cm|promedio_altura_cm|maximo_altura_cm|minimo_altura_cm|
+---------+-----------------+---------------+--------------+------------------+----------------+----------------+
|Verano   |222552           |170695         |29959627.0    |175.52            |226.0           |127.0           |
|Invierno |48564            |40250          |7027252.0     |174.59            |211.0           |137.0           |
+---------+-----------------+---------------+--------------+------------------+----------------+----------------+

Grupos de temporada: {'Verano', 'Invierno'} 


Participaciones conciliadas: 271,116 = 271,116 | mínimo <= promedio <= máximo: OK


**Interpretación — altura por temporada**

- Verano concentra 222.552 participaciones (82,1 %) e Invierno 48.564 (17,9 %). No aparecen otros grupos.
- La altura promedio es de 175,52 cm en Verano y de 174,59 cm en Invierno: menos de 1 cm de diferencia.
- El rango es más amplio en Verano (127 a 226 cm) que en Invierno (137 a 211 cm).
- La altura está registrada en 170.695 participaciones de Verano (76,7 %) y en 40.250 de Invierno (82,9 %). La suma de alturas en centímetros solo sirve para controlar los cálculos; no tiene una interpretación propia.

### 8.4 Edad por sexo (`sexo`)

Suma, promedio, máximo y mínimo de `edad` agrupados por el código `genero` y su `Descripción_sexo` (referenciada con comillas invertidas en SQL por el tilde). Si hubiera registros `No informado` se mantendrían como grupo propio; no se eliminan antes de la consulta.

In [32]:
sexo = spark.sql("""
    SELECT genero,
           `Descripción_sexo`,
           COUNT(*)               AS n_participaciones,
           COUNT(edad)            AS n_edad_valida,
           SUM(edad)              AS suma_edad,
           ROUND(AVG(edad), 2)    AS promedio_edad,
           MAX(edad)              AS maximo_edad,
           MIN(edad)              AS minimo_edad
    FROM participaciones_olimpicas
    GROUP BY genero, `Descripción_sexo`
    ORDER BY genero
""")
sexo.show(truncate=False)
verificar_agregacion(sexo, "promedio_edad", "minimo_edad", "maximo_edad")

+------+----------------+-----------------+-------------+---------+-------------+-----------+-----------+
|genero|Descripción_sexo|n_participaciones|n_edad_valida|suma_edad|promedio_edad|maximo_edad|minimo_edad|
+------+----------------+-----------------+-------------+---------+-------------+-----------+-----------+
|1     |Hombre          |196594           |187544       |4581905  |24.43        |97         |10         |
|2     |Mujer           |74522            |74098        |1619623  |21.86        |74         |11         |
+------+----------------+-----------------+-------------+---------+-------------+-----------+-----------+

Participaciones conciliadas: 271,116 = 271,116 | mínimo <= promedio <= máximo: OK


**Interpretación — edad por sexo**

- Los hombres concentran 196.594 participaciones (72,5 %) y las mujeres 74.522 (27,5 %). No hay un grupo `No informado`, porque `genero` solo trae los códigos 1 y 2.
- La edad promedio es de 24,43 años en hombres y de 21,86 en mujeres: 2,57 años de diferencia.
- El rango de edad de los hombres (10 a 97 años) es más amplio que el de las mujeres (11 a 74).
- La edad está registrada en el 95,4 % de las participaciones de hombres (187.544) y en el 99,4 % de las de mujeres (74.098).

## 9. Presentación mediante video

El video grupal `mcdi502_s2_grupo1_video.mp4`, de 7 a 10 minutos, se entrega junto a este notebook en un único archivo ZIP. Todos los integrantes participan y cada uno explica su parte con las salidas de este notebook:

| Integrante | Puntos que presenta |
|---|---|
| Yerko Gallardo | 1 a 3: entorno, RDDs, unión, DataFrame `deportista` y transformaciones |
| Sebastián Rojas | 4 a 6: DataFrames, integración, optimización, cinco particiones e inspección |
| José Ignacio Rodríguez | 7, 8 y conclusiones: IMC, `Descripción_sexo` y las cuatro agregaciones con Spark SQL |

## 10. Controles finales, conclusiones y cierre

Resumen de los objetos del bloque y de los controles que siguen vigentes en la sesión.

In [33]:
print(f"df_integrado   : {n_entrada:,} filas | {df_integrado.rdd.getNumPartitions()} particiones | {df_integrado.storageLevel}")
print(f"df_analisis    : {df_analisis.count():,} filas | {len(df_analisis.columns)} columnas (incluye IMC y Descripción_sexo)")
for nombre_df, tabla in [("medallas_equipo", medallas_equipo), ("edad_medalla", edad_medalla),
                         ("temporada", temporada), ("sexo", sexo)]:
    print(f"{nombre_df:15s}: {tabla.count():,} filas | columnas: {tabla.columns}")
print("Vistas temporales:", [t.name for t in spark.catalog.listTables()])

df_integrado   : 271,116 filas | 5 particiones | Disk Memory Serialized 1x Replicated


df_analisis    : 271,116 filas | 23 columnas (incluye IMC y Descripción_sexo)


medallas_equipo: 1,012 filas | columnas: ['equipo_id', 'equipo', 'oro', 'plata', 'bronce', 'total_medallas']
edad_medalla   : 4 filas | columnas: ['tipo_medalla', 'codigo_medalla', 'n_participaciones', 'n_edad_valida', 'suma_edad', 'promedio_edad', 'maximo_edad', 'minimo_edad']


temporada      : 2 filas | columnas: ['temporada', 'n_participaciones', 'n_altura_valida', 'suma_altura_cm', 'promedio_altura_cm', 'maximo_altura_cm', 'minimo_altura_cm']


sexo           : 2 filas | columnas: ['genero', 'Descripción_sexo', 'n_participaciones', 'n_edad_valida', 'suma_edad', 'promedio_edad', 'maximo_edad', 'minimo_edad']


Vistas temporales: ['participaciones_olimpicas']


**Conclusiones**

| Producto | Resultado | Control |
|---|---|---|
| `df_analisis` | 271.116 filas, 23 columnas; IMC en 206.853 participaciones (76,3 %) | Filas, columnas y `resultado_id` conservados; caso 180 cm / 80 kg = 24,69; inválidos quedan nulos |
| `medallas_equipo` | 1.012 equipos; oro, plata y bronce por equipo | Suma 13.372 / 13.116 / 13.295 = entrada |
| `edad_medalla` | 4 grupos (Oro, Plata, Bronce y Sin medalla) | Participaciones = 271.116; mínimo ≤ promedio ≤ máximo |
| `temporada` | Verano e Invierno, altura en cm | Participaciones = 271.116; mínimo ≤ promedio ≤ máximo |
| `sexo` | Hombre y Mujer | Participaciones = 271.116; mínimo ≤ promedio ≤ máximo |

- Las cuatro agregaciones se ejecutan con `spark.sql` sobre la vista `participaciones_olimpicas`. Las cantidades concilian con la entrada y ningún nulo se reemplazó por cero.
- Las cuatro medidas de edad casi no varían según el tipo de medalla (promedio de 23,7 años en los cuatro grupos), y la altura promedio difiere en menos de 1 cm entre Verano e Invierno. La diferencia más visible es por sexo: en promedio, las participaciones de mujeres corresponden a deportistas 2,57 años menores.
- Los registros de medalla por equipo se concentran en pocos países, pero no equivalen al medallero oficial: en las pruebas colectivas, cada integrante suma un registro.
- Todas las cifras están medidas por **participación**, no por persona. Quien participa en varias pruebas o ediciones pesa varias veces, y su edad, altura y peso se repiten. Medirlas por persona exigiría deduplicar por `deportista_id`, y eso cambiaría la pregunta analítica.
- La falta de datos limita sobre todo la altura y el peso (y por lo tanto el IMC): cerca de un cuarto de las participaciones no tienen estas medidas. La edad falta en el 3,5 % de las participaciones.

### 10.1 Fuentes y cierre de recursos

**Fuentes utilizadas**

- Archivos del curso en `data/`: `deportista.csv`, `deportista2.csv`, `resultados.csv`, `evento.csv`, `equipo.csv` y `juegos.json`.
- Documentación de Apache Spark 4.x: [PySpark API](https://spark.apache.org/docs/latest/api/python/index.html) y [Spark SQL](https://spark.apache.org/docs/latest/sql-ref.html) (agregaciones, `CASE WHEN`, vistas temporales).
- Definición de IMC como peso (kg) dividido por el cuadrado de la altura (m): Organización Mundial de la Salud, [*Obesity and overweight*](https://www.who.int/news-room/fact-sheets/detail/obesity-and-overweight).

**Cierre.** Se libera `df_integrado`, persistido en la sección 5, y se detiene la sesión de Spark. `spark.stop()` también libera las cachés de RDD que siguen activas desde las secciones 2 y 3.

In [34]:
print("RDDs persistidos antes del cierre:", len(sc._jsc.getPersistentRDDs()))
df_integrado.unpersist()
print("df_integrado sigue en caché:", df_integrado.is_cached)
spark.catalog.dropTempView("participaciones_olimpicas")
spark.stop()
print("Sesión de Spark detenida.")

RDDs persistidos antes del cierre: 5
df_integrado sigue en caché: False


Sesión de Spark detenida.
